# Levee Detection from Sentinel-1 SAR
## Feature Engineering → Training Data → XGBoost

**Stack:**
- Raster I/O: GDAL directly (`osgeo.gdal`)
- Vector I/O: GeoPandas + pyogrio
- Texture: scikit-image GLCM
- Model: XGBoost

**Expected input:**
- Preprocessed Sentinel-1 GeoTIFFs: sigma0 VV and VH, linear units, EPSG:2180, 10m
- pyroSAR output naming: `S1A__IW___A_YYYYMMDDTHHMMSS_VV_grd_elp.tif`
- BDOT10k GeoPackage with layer `OT_BUZM_L`

**Install dependencies:**
```
pip install numpy geopandas pyogrio scikit-image xgboost scikit-learn matplotlib tqdm
```

---
## Section 0 — Configuration

In [ ]:
from pathlib import Path

# ── Input directories ──────────────────────────────────────────────────────
# pyroSAR output directories — separate VV and VH GeoTIFFs per scene
ASC_DIR  = Path(r'C:/data/processed/ascending')
DESC_DIR = Path(r'C:/data/processed/descending')

# BDOT10k GeoPackage
BDOT10K_PATH = Path(r'C:/data/bdot10k/BDOT10k.gpkg')

# ── Output directory ───────────────────────────────────────────────────────
OUT_DIR = Path(r'C:/data/model')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Coordinate system ──────────────────────────────────────────────────────
EPSG       = 2180    # PL-1992 — consistent with BDOT10k
PIXEL_SIZE = 10.0   # metres

# ── GLCM parameters ────────────────────────────────────────────────────────
GLCM_WINDOW = 7     # sliding window size (pixels)
GLCM_LEVELS = 64    # grey levels for quantisation
GLCM_DIST   = 1     # pixel distance for co-occurrence

# ── Ground truth filter thresholds ────────────────────────────────────────
# wał przeciwpowodziowy lub grobla: no width filter
# nasyp: length >= 1000m, height >= 2m, crown width >= 12m
NASYP_MIN_LENGTH = 1000.0
NASYP_MIN_HEIGHT = 2.0
NASYP_MIN_WIDTH  = 12.0

# ── Levee rasterization buffer ────────────────────────────────────────────
LEVEE_BUFFER = 10.0  # metres — 1 pixel

# ── Sampling ──────────────────────────────────────────────────────────────
N_POSITIVE   = 50_000
N_NEGATIVE   = 50_000
RANDOM_STATE = 42

print('Configuration loaded.')
print(f'  ASC_DIR  : {ASC_DIR}')
print(f'  DESC_DIR : {DESC_DIR}')
print(f'  BDOT10K  : {BDOT10K_PATH}')
print(f'  OUT_DIR  : {OUT_DIR}')

---
## Section 1 — GDAL Utility Functions

In [ ]:
import numpy as np
from osgeo import gdal, ogr

gdal.UseExceptions()


def read_band(path: Path) -> tuple:
    """
    Reads a single-band GeoTIFF using GDAL.
    Returns (array float32, geo_info dict).
    NoData pixels are replaced with np.nan.
    """
    ds = gdal.Open(str(path), gdal.GA_ReadOnly)
    if ds is None:
        raise FileNotFoundError(f'Cannot open: {path}')

    band = ds.GetRasterBand(1)
    arr  = band.ReadAsArray().astype(np.float32)

    nodata = band.GetNoDataValue()
    if nodata is not None:
        arr[arr == nodata] = np.nan

    geo_info = {
        'geotransform': ds.GetGeoTransform(),
        'projection':   ds.GetProjection(),
        'nrows':        ds.RasterYSize,
        'ncols':        ds.RasterXSize,
    }
    ds = None
    return arr, geo_info


def write_multiband_tiff(
    path: Path,
    arrays: list,
    band_names: list,
    geo_info: dict,
    nodata: float = -9999.0,
):
    """
    Writes a list of 2D arrays as a multiband GeoTIFF.
    LZW compressed, BigTIFF if needed.
    NaN values replaced with nodata before writing.
    """
    nrows, ncols = arrays[0].shape
    nbands = len(arrays)

    driver = gdal.GetDriverByName('GTiff')
    ds = driver.Create(
        str(path), ncols, nrows, nbands, gdal.GDT_Float32,
        options=['COMPRESS=LZW', 'BIGTIFF=IF_SAFER', 'TILED=YES']
    )
    ds.SetGeoTransform(geo_info['geotransform'])
    ds.SetProjection(geo_info['projection'])

    for i, (arr, name) in enumerate(zip(arrays, band_names), start=1):
        out  = np.where(np.isnan(arr), nodata, arr).astype(np.float32)
        band = ds.GetRasterBand(i)
        band.WriteArray(out)
        band.SetNoDataValue(nodata)
        band.SetDescription(name)

    ds.FlushCache()
    ds = None
    print(f'  Saved: {path.name}  ({nbands} bands, {nrows}x{ncols})')


print('Utility functions defined.')

---
## Section 2 — Multi-temporal Averaging (xarray + rioxarray)

All scenes loaded as `xr.DataArray` via `rioxarray.open_rasterio()`.
Grid alignment uses `reindex_like()` to a **union grid** (maximum spatial extent
across all scenes) — faster and more readable than GDAL in-memory warp.

Pixels outside any individual scene are `NaN` — `mean(skipna=True)` ignores them,
so the temporal average is valid even at the edges where not all scenes overlap.

Averaging in **linear units** (not dB) — Jensen's inequality.
The union reference grid is established from ASC VV and reused for all
subsequent stacks (ASC VH, DESC VV, DESC VH) to guarantee a consistent
18-band feature stack with identical spatial dimensions.

**Install:** `pip install rioxarray`

In [ ]:
import numpy as np
from osgeo import gdal, ogr

gdal.UseExceptions()


def read_band(path: Path) -> tuple:
    """
    Reads a single-band GeoTIFF using GDAL.
    Returns (array float32, geo_info dict).
    NoData pixels are replaced with np.nan.
    """
    ds = gdal.Open(str(path), gdal.GA_ReadOnly)
    if ds is None:
        raise FileNotFoundError(f'Cannot open: {path}')

    band = ds.GetRasterBand(1)
    arr  = band.ReadAsArray().astype(np.float32)

    nodata = band.GetNoDataValue()
    if nodata is not None:
        arr[arr == nodata] = np.nan

    geo_info = {
        'geotransform': ds.GetGeoTransform(),
        'projection':   ds.GetProjection(),
        'nrows':        ds.RasterYSize,
        'ncols':        ds.RasterXSize,
    }
    ds = None
    return arr, geo_info


def write_multiband_tiff(
    path: Path,
    arrays: list,
    band_names: list,
    geo_info: dict,
    nodata: float = -9999.0,
):
    """
    Writes a list of 2D arrays as a multiband GeoTIFF.
    LZW compressed, BigTIFF if needed.
    NaN values replaced with nodata before writing.
    """
    nrows, ncols = arrays[0].shape
    nbands = len(arrays)

    driver = gdal.GetDriverByName('GTiff')
    ds = driver.Create(
        str(path), ncols, nrows, nbands, gdal.GDT_Float32,
        options=['COMPRESS=LZW', 'BIGTIFF=IF_SAFER', 'TILED=YES']
    )
    ds.SetGeoTransform(geo_info['geotransform'])
    ds.SetProjection(geo_info['projection'])

    for i, (arr, name) in enumerate(zip(arrays, band_names), start=1):
        out  = np.where(np.isnan(arr), nodata, arr).astype(np.float32)
        band = ds.GetRasterBand(i)
        band.WriteArray(out)
        band.SetNoDataValue(nodata)
        band.SetDescription(name)

    ds.FlushCache()
    ds = None
    print(f'  Saved: {path.name}  ({nbands} bands, {nrows}x{ncols})')


print('Utility functions defined.')

---
## Section 3 — Feature Engineering
### 3.1 VH/VV Ratio
Computed in linear units. Encodes the cross-polarisation to co-polarisation relationship,
which is sensitive to geometric structure and volume scattering — both relevant for levee detection.

In [ ]:
def compute_ratio(vh: np.ndarray, vv: np.ndarray) -> np.ndarray:
    """
    Computes VH/VV backscatter ratio in linear units.
    Division by zero returns NaN.
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(vv > 0, vh / vv, np.nan)
    return ratio.astype(np.float32)


asc_ratio  = compute_ratio(asc_vh_mean,  asc_vv_mean)
desc_ratio = compute_ratio(desc_vh_mean, desc_vv_mean)

print('VH/VV ratio computed.')
print(f'  ASC  ratio — mean: {np.nanmean(asc_ratio):.4f}')
print(f'  DESC ratio — mean: {np.nanmean(desc_ratio):.4f}')

### 3.2 GLCM Texture Features — PyTorch (GPU accelerated)

Computed on multi-temporal mean VV (highest SNR band).
Features: **contrast**, **homogeneity**, **energy**, **correlation**.
Four angles [0°, 45°, 90°, 135°] averaged — rotation-invariant descriptor.

Implementation uses `torch.unfold` + `scatter_add_` — all patches processed in parallel on GPU.
Falls back to CPU automatically if CUDA is not available.
Batch size controls GPU memory usage (default: 4096 patches per batch).

In [ ]:
import torch
import torch.nn.functional as F


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


def compute_glcm_pytorch(
    arr: np.ndarray,
    window: int = GLCM_WINDOW,
    levels: int = GLCM_LEVELS,
    distance: int = GLCM_DIST,
    batch_size: int = 4096,
    device: str = DEVICE,
) -> dict:
    """
    GLCM texture features via PyTorch — GPU accelerated.

    Strategy:
      1. torch.unfold extracts all (window x window) patches at once.
      2. For each of 4 angles, reference and neighbour pixels are extracted
         via slice indexing — fully vectorized, no Python loops over pixels.
      3. scatter_add_ accumulates co-occurrence counts into GLCM matrices
         for all patches in a batch simultaneously.
      4. GLCM properties (contrast, homogeneity, energy, correlation)
         computed as tensor dot products.

    Processing in batches avoids OOM on large scenes.
    Falls back to CPU if CUDA is not available.
    """
    nrows, ncols = arr.shape
    N   = nrows * ncols
    pad = window // 2
    d   = distance

    # Quantise image to [0, levels-1]
    q = quantise(arr, levels)  # uint8 numpy array

    # Move to device
    t = torch.from_numpy(q.astype(np.int64)).to(device)

    # Reflect-pad
    t_pad = F.pad(
        t.float().unsqueeze(0).unsqueeze(0),
        (pad, pad, pad, pad),
        mode='reflect',
    ).squeeze().long()  # (H + 2*pad, W + 2*pad)

    # Extract all patches via unfold: (nrows, ncols, window, window)
    patches = t_pad.unfold(0, window, 1).unfold(1, window, 1)
    patches = patches.reshape(N, window, window)  # (N, W, W)

    # Angle pair slices (row_ref, row_neigh, col_ref, col_neigh)
    # Four directions — averaged for rotation invariance
    angle_pairs = [
        (slice(None),    slice(None),    slice(None, -d), slice(d, None)),  # 0°
        (slice(d, None), slice(None,-d), slice(None, -d), slice(d, None)),  # 45°
        (slice(None,-d), slice(d, None), slice(None),     slice(None)),     # 90°
        (slice(None,-d), slice(d, None), slice(None, -d), slice(d, None)),  # 135°
    ]

    # Index grids for GLCM property computation
    i_idx = torch.arange(levels, dtype=torch.float32, device=device)
    j_idx = torch.arange(levels, dtype=torch.float32, device=device)
    I, J  = torch.meshgrid(i_idx, j_idx, indexing='ij')  # (L, L)
    diff  = I - J

    contrast_w    = diff ** 2
    homogeneity_w = 1.0 / (1.0 + diff.abs())

    # Output accumulators (stay on device until end)
    out_contrast    = torch.zeros(N, dtype=torch.float32, device=device)
    out_homogeneity = torch.zeros(N, dtype=torch.float32, device=device)
    out_energy      = torch.zeros(N, dtype=torch.float32, device=device)
    out_correlation = torch.zeros(N, dtype=torch.float32, device=device)

    n_batches = (N + batch_size - 1) // batch_size

    for b in tqdm(range(n_batches), desc=f'  GLCM [{device}]'):
        start = b * batch_size
        end   = min(start + batch_size, N)
        B     = end - start

        pb = patches[start:end]  # (B, W, W)

        # Accumulate co-occurrences into GLCM: (B, L, L)
        glcm = torch.zeros(B, levels, levels, dtype=torch.float32, device=device)
        gf   = glcm.view(B, -1)  # (B, L*L) — view for scatter

        for dy_r, dy_n, dx_r, dx_n in angle_pairs:
            ref   = pb[:, dy_r, dx_r].reshape(B, -1)  # (B, M)
            neigh = pb[:, dy_n, dx_n].reshape(B, -1)
            M     = ref.shape[1]

            ones      = torch.ones(B, M, dtype=torch.float32, device=device)
            idx       = (ref   * levels + neigh).long()  # (B, M)
            idx_sym   = (neigh * levels + ref  ).long()

            # Symmetric GLCM: count both (i,j) and (j,i)
            gf.scatter_add_(1, idx,     ones)
            gf.scatter_add_(1, idx_sym, ones)

        # Normalise
        g = glcm / glcm.sum(dim=(1, 2), keepdim=True).clamp(min=1e-10)

        # Contrast
        out_contrast[start:end] = (g * contrast_w).sum(dim=(1, 2))

        # Homogeneity
        out_homogeneity[start:end] = (g * homogeneity_w).sum(dim=(1, 2))

        # Energy
        out_energy[start:end] = (g ** 2).sum(dim=(1, 2))

        # Correlation
        mu_i  = (g * I).sum(dim=(1, 2))           # (B,)
        mu_j  = (g * J).sum(dim=(1, 2))
        var_i = (g * (I - mu_i.view(-1, 1, 1)) ** 2).sum(dim=(1, 2))
        var_j = (g * (J - mu_j.view(-1, 1, 1)) ** 2).sum(dim=(1, 2))
        std   = (var_i * var_j).sqrt().clamp(min=1e-10)
        num   = (g * (I - mu_i.view(-1, 1, 1)) * (J - mu_j.view(-1, 1, 1))).sum(dim=(1, 2))
        out_correlation[start:end] = num / std

    def to2d(t):
        return t.cpu().numpy().reshape(nrows, ncols).astype(np.float32)

    return {
        'contrast':    to2d(out_contrast),
        'homogeneity': to2d(out_homogeneity),
        'energy':      to2d(out_energy),
        'correlation': to2d(out_correlation),
    }


print('Computing GLCM — ascending VV...')
asc_glcm = compute_glcm_pytorch(asc_vv_mean)

print('Computing GLCM — descending VV...')
desc_glcm = compute_glcm_pytorch(desc_vv_mean)

print('GLCM features computed.')


### 3.3 Feature Stack Assembly

18 bands total (9 ascending + 9 descending):
VV mean, VH mean, VV std, VH std, VH/VV ratio, GLCM contrast, homogeneity, energy, correlation.

In [ ]:
BAND_NAMES = [
    'asc_vv_mean', 'asc_vh_mean', 'asc_vv_std', 'asc_vh_std', 'asc_ratio',
    'asc_glcm_contrast', 'asc_glcm_homogeneity', 'asc_glcm_energy', 'asc_glcm_correlation',
    'desc_vv_mean', 'desc_vh_mean', 'desc_vv_std', 'desc_vh_std', 'desc_ratio',
    'desc_glcm_contrast', 'desc_glcm_homogeneity', 'desc_glcm_energy', 'desc_glcm_correlation',
]

FEATURE_ARRAYS = [
    asc_vv_mean, asc_vh_mean, asc_vv_std, asc_vh_std, asc_ratio,
    asc_glcm['contrast'], asc_glcm['homogeneity'], asc_glcm['energy'], asc_glcm['correlation'],
    desc_vv_mean, desc_vh_mean, desc_vv_std, desc_vh_std, desc_ratio,
    desc_glcm['contrast'], desc_glcm['homogeneity'], desc_glcm['energy'], desc_glcm['correlation'],
]

assert len(BAND_NAMES) == len(FEATURE_ARRAYS) == 18

FEATURE_STACK_PATH = OUT_DIR / 'feature_stack.tif'
write_multiband_tiff(
    path=FEATURE_STACK_PATH,
    arrays=FEATURE_ARRAYS,
    band_names=BAND_NAMES,
    geo_info=geo_ref,
)
print(f'Feature stack: {FEATURE_STACK_PATH}')

---
## Section 4 — Ground Truth from BDOT10k

Two object types from `OT_BUZM_L`:
- `wał przeciwpowodziowy lub grobla` — all records, no geometric filter
- `nasyp` — filtered: length ≥ 1000 m, height ≥ 2 m, crown width ≥ 12 m

In [ ]:
import geopandas as gpd


def load_ground_truth(bdot10k_path: Path, target_epsg: int = EPSG) -> gpd.GeoDataFrame:
    """
    Loads and filters levee ground truth from BDOT10k OT_BUZM_L layer.
    Returns GeoDataFrame in target CRS (EPSG:2180).
    """
    print(f'Loading BDOT10k: {bdot10k_path}')
    gdf = gpd.read_file(bdot10k_path, layer='OT_BUZM_L', engine='pyogrio')
    print(f'  Total OT_BUZM_L features: {len(gdf)}')
    print(f'  CRS: {gdf.crs}')

    # Filter 1: wał przeciwpowodziowy lub grobla — all records
    hraze = gdf[gdf['rodzaj'] == 'wal przeciwpowodziowy lub grobla'].copy()
    hraze['source'] = 'hraze'
    print(f'  Hráze (wał + grobla): {len(hraze)}')

    # Filter 2: nasyp — length, height, crown width
    if gdf.crs and gdf.crs.is_projected:
        lengths = gdf.geometry.length
    else:
        lengths = gdf.geometry.to_crs(epsg=target_epsg).length

    nasypy = gdf[
        (gdf['rodzaj'] == 'nasyp') &
        (lengths >= NASYP_MIN_LENGTH) &
        (gdf['wysokosc'].fillna(0)   >= NASYP_MIN_HEIGHT) &
        (gdf['szerKorony'].fillna(0) >= NASYP_MIN_WIDTH)
    ].copy()
    nasypy['source'] = 'nasyp'
    print(f'  Náspy (filtered): {len(nasypy)}')

    # Combine
    gt = gpd.GeoDataFrame(
        gpd.pd.concat([hraze, nasypy], ignore_index=True),
        crs=gdf.crs,
    )
    print(f'  Combined: {len(gt)} features')

    # Reproject if needed
    if gt.crs.to_epsg() != target_epsg:
        gt = gt.to_crs(epsg=target_epsg)
        print(f'  Reprojected to EPSG:{target_epsg}')

    return gt


ground_truth = load_ground_truth(BDOT10K_PATH)
print(ground_truth[['rodzaj', 'source', 'szerKorony', 'wysokosc']].head(10))

### 4.1 Rasterize Ground Truth

In [ ]:
import tempfile


def rasterize_ground_truth(gdf: gpd.GeoDataFrame, geo_info: dict) -> np.ndarray:
    """
    Rasterizes levee vectors to a binary mask matching the feature stack grid.
    Line geometries are buffered by LEVEE_BUFFER metres before rasterization.
    Returns uint8 array: 1 = levee, 0 = non-levee.
    """
    gt    = geo_info['geotransform']
    proj  = geo_info['projection']
    nrows = geo_info['nrows']
    ncols = geo_info['ncols']

    # Buffer line geometries
    gdf_buf = gdf.copy()
    gdf_buf['geometry'] = gdf_buf.geometry.buffer(LEVEE_BUFFER)

    # Write to temp GeoPackage for GDAL rasterization
    with tempfile.NamedTemporaryFile(suffix='.gpkg', delete=False) as tmp:
        tmp_path = Path(tmp.name)
    gdf_buf.to_file(tmp_path, driver='GPKG', engine='pyogrio')

    # Open vector layer
    vec_ds  = ogr.Open(str(tmp_path))
    vec_lyr = vec_ds.GetLayer(0)

    # Create in-memory raster
    mem_drv = gdal.GetDriverByName('MEM')
    out_ds  = mem_drv.Create('', ncols, nrows, 1, gdal.GDT_Byte)
    out_ds.SetGeoTransform(gt)
    out_ds.SetProjection(proj)

    out_band = out_ds.GetRasterBand(1)
    out_band.Fill(0)
    out_band.SetNoDataValue(255)

    # Burn value 1 for all levee features
    gdal.RasterizeLayer(out_ds, [1], vec_lyr, burn_values=[1])
    out_ds.FlushCache()

    mask = out_band.ReadAsArray().astype(np.uint8)

    out_ds = None
    vec_ds = None
    tmp_path.unlink(missing_ok=True)

    print(f'  Positive pixels (levee): {int(mask.sum()):,}')
    print(f'  Negative pixels (other): {int((mask == 0).sum()):,}')
    return mask


print('Rasterizing ground truth...')
label_mask = rasterize_ground_truth(ground_truth, geo_ref)

# Save for QC in QGIS
label_path = OUT_DIR / 'label_mask.tif'
write_multiband_tiff(
    path=label_path,
    arrays=[label_mask.astype(np.float32)],
    band_names=['label'],
    geo_info=geo_ref,
    nodata=255,
)
print(f'Label mask saved -> {label_path}')

---
## Section 5 — Stratified Pixel Sampling

In [ ]:
from sklearn.model_selection import train_test_split


def build_feature_matrix(feature_arrays: list, label_mask: np.ndarray) -> tuple:
    """
    Builds X (feature matrix) and y (labels) from stratified pixel sampling.
    Excludes pixels where any feature band is NaN.
    Returns (X float32, y int8).
    """
    rng = np.random.default_rng(RANDOM_STATE)

    feature_cube = np.stack(feature_arrays, axis=-1)           # (nrows, ncols, 18)
    X_flat       = feature_cube.reshape(-1, len(feature_arrays))  # (N, 18)
    y_flat       = label_mask.ravel()                              # (N,)

    # Valid pixels: no NaN in any band
    valid = ~np.any(np.isnan(X_flat), axis=1)
    print(f'  Valid pixels: {valid.sum():,} / {len(valid):,}')

    pos_idx = np.where((y_flat == 1) & valid)[0]
    neg_idx = np.where((y_flat == 0) & valid)[0]
    print(f'  Available positive: {len(pos_idx):,}')
    print(f'  Available negative: {len(neg_idx):,}')

    n_pos = min(N_POSITIVE, len(pos_idx))
    n_neg = min(N_NEGATIVE, len(neg_idx))

    idx = np.concatenate([
        rng.choice(pos_idx, size=n_pos, replace=False),
        rng.choice(neg_idx, size=n_neg, replace=False),
    ])
    rng.shuffle(idx)

    X = X_flat[idx].astype(np.float32)
    y = y_flat[idx].astype(np.int8)
    print(f'  Sampled: {n_pos:,} positive + {n_neg:,} negative = {len(X):,} total')
    return X, y


print('Building feature matrix...')
X, y = build_feature_matrix(FEATURE_ARRAYS, label_mask)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Class balance — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}')

---
## Section 6 — XGBoost Training

In [ ]:
import xgboost as xgb
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
)
import matplotlib.pyplot as plt


scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',        # area under PR curve — robust for imbalanced data
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',         # fast histogram-based algorithm
)

print('Training XGBoost...')
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)
print(f'Best iteration: {model.best_iteration}')

### 6.1 Threshold Optimisation via PR Curve

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

# Optimal threshold: maximise F1
f1_scores = 2 * precision * recall / (precision + recall + 1e-10)
best_idx  = np.argmax(f1_scores[:-1])
best_thr  = float(thresholds[best_idx])
best_f1   = float(f1_scores[best_idx])

print(f'Optimal threshold : {best_thr:.3f}')
print(f'Best F1           : {best_f1:.3f}')
print(f'AP score          : {average_precision_score(y_test, y_proba):.3f}')
print(f'ROC-AUC           : {roc_auc_score(y_test, y_proba):.3f}')

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, lw=2, label='PR curve')
ax.axvline(recall[best_idx], color='red', linestyle='--', alpha=0.7,
           label=f'Optimal thr={best_thr:.2f} (F1={best_f1:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Levee Detection')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'pr_curve.png', dpi=150)
plt.show()

### 6.2 Classification Report

In [ ]:
y_pred = (y_proba >= best_thr).astype(int)
print(classification_report(
    y_test, y_pred,
    target_names=['non-levee', 'levee'],
    digits=3,
))

### 6.3 Feature Importance

In [ ]:
importance = model.feature_importances_
sorted_idx = np.argsort(importance)[::-1]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(
    [BAND_NAMES[i] for i in sorted_idx[::-1]],
    importance[sorted_idx[::-1]],
)
ax.set_xlabel('Feature importance (gain)')
ax.set_title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_importance.png', dpi=150)
plt.show()

print('Feature importance ranking:')
for rank, i in enumerate(sorted_idx, 1):
    print(f'  {rank:2d}. {BAND_NAMES[i]:<35s} {importance[i]:.4f}')

---
## Section 7 — Save Model and Metadata

In [ ]:
import json
from datetime import datetime
from sklearn.metrics import average_precision_score, roc_auc_score

model_path = OUT_DIR / 'xgb_levee_model.json'
model.save_model(str(model_path))
print(f'Model saved -> {model_path}')

metadata = {
    'created':            datetime.now().isoformat(),
    'model':              'XGBoostClassifier',
    'best_iteration':     int(model.best_iteration),
    'optimal_threshold':  best_thr,
    'best_f1':            best_f1,
    'ap_score':           float(average_precision_score(y_test, y_proba)),
    'roc_auc':            float(roc_auc_score(y_test, y_proba)),
    'band_names':         BAND_NAMES,
    'n_train':            int(len(X_train)),
    'n_test':             int(len(X_test)),
    'epsg':               EPSG,
    'pixel_size_m':       PIXEL_SIZE,
    'glcm_window':        GLCM_WINDOW,
    'glcm_levels':        GLCM_LEVELS,
    'levee_buffer_m':     LEVEE_BUFFER,
    'nasyp_min_length_m': NASYP_MIN_LENGTH,
    'nasyp_min_height_m': NASYP_MIN_HEIGHT,
    'nasyp_min_width_m':  NASYP_MIN_WIDTH,
}

meta_path = OUT_DIR / 'model_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved -> {meta_path}')

print('\n=== Training complete ===')
print(f"  Threshold : {best_thr:.3f}")
print(f"  AP score  : {metadata['ap_score']:.3f}")
print(f"  ROC-AUC   : {metadata['roc_auc']:.3f}")